In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

df_full = pd.read_csv('features_v3_ohe_crop.csv')

In [2]:
# Model A: General Crops (everything EXCEPT sugarcane)
df_general = df_full[df_full['crop_sugarcane'] == 0].copy()

# Model B: Sugarcane Only
df_sugarcane = df_full[df_full['crop_sugarcane'] == 1].copy()

print(f"General crop samples: {len(df_general)}")
print(f"Sugarcane samples: {len(df_sugarcane)}")

General crop samples: 15520
Sugarcane samples: 633


In [3]:
print("\n--- Training Model A: General Crops ---")

# We can now drop ALL crop columns, as 'crop_sugarcane' is 0
# and the other 'crop_*' features will help distinguish between the remaining crops.
X_gen = df_general.drop(columns=['yield_log1p'])
y_gen = df_general['yield_log1p']

X_train_gen, X_test_gen, y_train_gen, y_test_gen = train_test_split(X_gen, y_gen, test_size=0.2, random_state=42)

rf_general = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_general.fit(X_train_gen, y_train_gen)

# Evaluate Model A
y_pred_gen = rf_general.predict(X_test_gen)
rmse_gen = np.sqrt(mean_squared_error(y_test_gen, y_pred_gen))
r2_gen = r2_score(y_test_gen, y_pred_gen)

print(f"General Model RMSE: {rmse_gen:.4f}")
print(f"General Model R²:     {r2_gen:.4f}")


--- Training Model A: General Crops ---
General Model RMSE: 0.1949
General Model R²:     0.8985


In [4]:
print("\n--- Training Model B: Sugarcane ---")

# For this model, ALL 'crop_*' columns are useless.
# 'crop_sugarcane' is always 1, and all others are 0. They have zero variance.
# We MUST drop them.
crop_cols_to_drop = X_gen.filter(like='crop_').columns
X_sugar = df_sugarcane.drop(columns=['yield_log1p'] + list(crop_cols_to_drop))
y_sugar = df_sugarcane['yield_log1p']

X_train_sugar, X_test_sugar, y_train_sugar, y_test_sugar = train_test_split(X_sugar, y_sugar, test_size=0.2, random_state=42)

rf_sugarcane = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_sugarcane.fit(X_train_sugar, y_train_sugar)

# Evaluate Model B
y_pred_sugar = rf_sugarcane.predict(X_test_sugar)
rmse_sugar = np.sqrt(mean_squared_error(y_test_sugar, y_pred_sugar))
r2_sugar = r2_score(y_test_sugar, y_pred_sugar)

print(f"Sugarcane Model RMSE: {rmse_sugar:.4f}")
print(f"Sugarcane Model R²:     {r2_sugar:.4f}")


--- Training Model B: Sugarcane ---
Sugarcane Model RMSE: 0.2157
Sugarcane Model R²:     0.2594


In [5]:
# Feature Importance for General Model
importances_gen = rf_general.feature_importances_
features_gen = X_train_gen.columns
df_imp_gen = pd.DataFrame({'feature': features_gen, 'importance': importances_gen}).sort_values(by='importance', ascending=False)

print("\n--- Top 15 Features: General Model ---")
print(df_imp_gen.head(15))

# Feature Importance for Sugarcane Model
importances_sugar = rf_sugarcane.feature_importances_
features_sugar = X_train_sugar.columns
df_imp_sugar = pd.DataFrame({'feature': features_sugar, 'importance': importances_sugar}).sort_values(by='importance', ascending=False)

print("\n--- Top 15 Features: Sugarcane Model ---")
print(df_imp_sugar.head(15))

# Save the models
joblib.dump(rf_general, 'general_yield_model.pkl')
joblib.dump(X_gen.columns.tolist(), 'general_model_columns.pkl')

joblib.dump(rf_sugarcane, 'sugarcane_yield_model.pkl')
joblib.dump(X_sugar.columns.tolist(), 'sugarcane_model_columns.pkl')

print("\nSpecialized models and column lists saved.")


--- Top 15 Features: General Model ---
              feature  importance
63        crop_potato    0.335473
70  crop_sweet_potato    0.114696
72         crop_onion    0.109161
55          crop_rice    0.045285
3     total_precip_mm    0.043523
1                area    0.043099
60         crop_maize    0.037393
0                year    0.036523
75         crop_other    0.034571
73        crop_garlic    0.034285
57     crop_groundnut    0.032971
66         crop_wheat    0.021663
2          avg_temp_c    0.020344
49      season_Autumn    0.013902
6              mean_p    0.012186

--- Top 15 Features: Sugarcane Model ---
                     feature  importance
1                       area    0.260884
3            total_precip_mm    0.155421
2                 avg_temp_c    0.124536
0                       year    0.084301
31          district_jajapur    0.023826
40       district_mayurbhanj    0.023116
41      district_nabarangpur    0.021978
48       district_sundargarh    0.020722
43   